# Benchmark: DRNN + Optuna (Direct Competitor)

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## Purpose

This notebook trains the **identical DRNN architecture** as the proposed model,  
but replaces NiOA with **Optuna** (Tree-structured Parzen Estimator, TPE)  
for hyperparameter optimisation.

### Fairness constraints enforced
| Constraint | Value |
|---|---|
| Architecture | Identical to NiOA-DRNN |
| Search space | Identical |
| Optimisation subset | 30 % of training data |
| Trials budget | `N_AGENTS × (MAX_ITERATIONS + 1)` = 42 |
| Time limit per trial | Same `OPT_TIME_LIMIT` seconds |
| Splits | Identical canonical frozen splits |
| Evaluation metrics | Identical |

### v2 changes
- **OOM fix**: final model evaluation uses batched prediction via  
  `SequenceDataGenerator` instead of `evaluate_keras_model(X_test)` which  
  passes the full 3.28 GB array to the GPU at once.
- Model, scaler, Optuna study, best params, training config all saved for  
  full reproducibility.
- Plots produced in original kWh space.


In [ ]:
import os, sys, json, gc, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
import tensorflow.keras.backend as K
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise ImportError('Run:  pip install optuna')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config  import (
    SPLITS_DIR, BENCHMARK_DIR, RESULTS_DIR,
    N_AGENTS, MAX_ITERATIONS, OPT_SUBSET_RATIO,
    OPT_EPOCHS, OPT_PATIENCE, OPT_TIME_LIMIT,
    FINAL_EPOCHS, FINAL_PATIENCE,
    RANDOM_SEED, FORECAST_HORIZONS,
    TARGET_LOG_TRANSFORM, LOSS_FUNCTION, HUBER_DELTA,
)
from core.utils    import configure_gpu, set_seeds, SequenceDataGenerator, to_python_types
from core.models   import create_lstm_model
from core.train    import TimeLimitCallback
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import compute_metrics, save_benchmark_results, build_comparison_table

configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style='whitegrid')

ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f'TensorFlow   : {tf.__version__}')
print(f'Optuna       : {optuna.__version__}')
print(f'Project root : {PROJECT_ROOT}')
print(f'log_transform: {TARGET_LOG_TRANSFORM}')
print(f'Loss         : {LOSS_FUNCTION}  delta={HUBER_DELTA}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU(s) found : {len(gpus)}  →  {[g.name for g in gpus]}')

In [ ]:
# ── Edit these lines only ───────────────────────────────────────────────────
HORIZONS_TO_RUN  = [60, 300, 900, 1800]
SKIP_COMPLETED   = False
FINAL_BATCH_SIZE = 16   # reduce to 8 if GPU has < 4 GB VRAM
# ───────────────────────────────────────────────────────────────────────────

# Equal function-evaluation budget to NiOA:
# NiOA: N_AGENTS initial + N_AGENTS × MAX_ITERATIONS refinements = 42 total
N_TRIALS = N_AGENTS * (1 + MAX_ITERATIONS)

for _h in HORIZONS_TO_RUN:
    assert _h in FORECAST_HORIZONS

print(f'Horizons queued   : {HORIZONS_TO_RUN}')
print(f'Optuna trials     : {N_TRIALS}  (= NiOA function evaluations)')
print(f'Final batch size  : {FINAL_BATCH_SIZE}')
print(f'OPT_TIME_LIMIT    : {OPT_TIME_LIMIT} s per trial')

---
## Main Loop — Optuna study + final training per horizon

In [ ]:
all_results = {}
loop_start  = time.time()

for _loop_idx, HORIZON in enumerate(HORIZONS_TO_RUN):

    _elapsed = (time.time() - loop_start) / 3600
    print(f'\n' + '=' * 60)
    print(f'  Horizon {_loop_idx+1}/{len(HORIZONS_TO_RUN)} : '
          f'k = {HORIZON} s   {_elapsed:.2f} h elapsed')
    print('=' * 60)

    if SKIP_COMPLETED:
        _m_path = os.path.join(
            BENCHMARK_DIR, f'horizon_{HORIZON}', 'drnn_optuna', 'metrics.json'
        )
        if os.path.isfile(_m_path):
            print('  [SKIP] drnn_optuna already evaluated.')
            continue

    # ── Output directories ───────────────────────────────────────────────────
    _opt_dir = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}', 'drnn_optuna')
    _pd      = os.path.join(_opt_dir, 'plots')
    for _d in [_opt_dir, _pd]:
        os.makedirs(_d, exist_ok=True)

    # ── Load splits ─────────────────────────────────────────────────────────
    print('  Loading splits ...')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
        load_splits(SPLITS_DIR, HORIZON)

    SEQ_LEN   = X_train.shape[1]
    NUM_FEATS = X_train.shape[2]

    _n_tr_opt = int(len(X_train) * OPT_SUBSET_RATIO)
    _n_va_opt = int(len(X_val)   * OPT_SUBSET_RATIO)

    # Freeze copies to avoid closure bugs in the Optuna objective
    _Xtr = X_train[:_n_tr_opt].copy()
    _ytr = y_train[:_n_tr_opt].copy()
    _Xva = X_val[:_n_va_opt].copy()
    _yva = y_val[:_n_va_opt].copy()

    print(f'  seq={SEQ_LEN}  feats={NUM_FEATS}  '
          f'opt_train={_n_tr_opt:,}  opt_val={_n_va_opt:,}')

    # ── Optuna objective ─────────────────────────────────────────────────────
    def _optuna_objective(
        trial,
        Xtr=_Xtr, ytr=_ytr, Xva=_Xva, yva=_yva,
        seq=SEQ_LEN, feats=NUM_FEATS,
    ):
        params = {
            'lstm_layers'   : trial.suggest_int('lstm_layers', 2, 3),
            'units'         : trial.suggest_categorical('units', [64, 128]),
            'dropout'       : trial.suggest_float('dropout', 0.3, 0.6),
            'optimizer'     : 'adamw',
            'learning_rate' : trial.suggest_float('learning_rate', 5e-5, 5e-4, log=True),
            'batch_size'    : 32,
        }
        _model = None
        try:
            _model = create_lstm_model(
                params, seq_len=seq, num_feats=feats,
                loss_fn=LOSS_FUNCTION, huber_delta=HUBER_DELTA,
            )
            _tr_gen = SequenceDataGenerator(
                Xtr, ytr, batch_size=params['batch_size'], shuffle=True
            )
            _va_gen = SequenceDataGenerator(
                Xva, yva, batch_size=params['batch_size'], shuffle=False
            )
            _hist = _model.fit(
                _tr_gen,
                validation_data = _va_gen,
                epochs          = OPT_EPOCHS,
                callbacks       = [
                    EarlyStopping(monitor='val_loss', patience=OPT_PATIENCE,
                                  restore_best_weights=True, verbose=0),
                    TimeLimitCallback(max_seconds=OPT_TIME_LIMIT),
                ],
                verbose=0,
            )
            _val_loss = float(np.min(_hist.history['val_loss']))
        except Exception as _e:
            print(f'  [Trial {trial.number}] failed — {_e}')
            _val_loss = float('inf')
        finally:
            if _model is not None:
                del _model
            K.clear_session();  gc.collect()
        return _val_loss

    # ── Run Optuna study ─────────────────────────────────────────────────────
    print(f'  Running {N_TRIALS} Optuna trials ...')
    _study = optuna.create_study(
        direction  = 'minimize',
        sampler    = optuna.samplers.TPESampler(seed=RANDOM_SEED),
        study_name = f'DRNN_Optuna_k{HORIZON}',
    )
    _study.optimize(_optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

    _best_p = dict(_study.best_params)
    _best_p['optimizer']  = 'adamw'
    _best_p['batch_size'] = 32

    print(f'  Best Optuna val loss : {_study.best_value:.8f}')
    print(f'  Best params : {_best_p}')

    # Save Optuna study results
    _optuna_trials_csv = os.path.join(_opt_dir, f'optuna_trials_k{HORIZON}.csv')
    _study.trials_dataframe().to_csv(_optuna_trials_csv, index=False)
    with open(os.path.join(_opt_dir, 'best_params.json'), 'w') as _f:
        json.dump(to_python_types(_best_p), _f, indent=4)

    del _Xtr, _ytr, _Xva, _yva
    gc.collect()

    # ── Final training ───────────────────────────────────────────────────────
    print(f'  Final training  batch={FINAL_BATCH_SIZE}  '
          f'epochs={FINAL_EPOCHS}  patience={FINAL_PATIENCE} ...')
    K.clear_session();  gc.collect()

    _final_model = create_lstm_model(
        _best_p, SEQ_LEN, NUM_FEATS,
        loss_fn=LOSS_FUNCTION, huber_delta=HUBER_DELTA,
    )
    _final_model.summary()

    _tr_gen_f = SequenceDataGenerator(
        X_train, y_train, batch_size=FINAL_BATCH_SIZE, shuffle=True, seed=RANDOM_SEED
    )
    _va_gen_f = SequenceDataGenerator(
        X_val, y_val, batch_size=FINAL_BATCH_SIZE, shuffle=False
    )

    _hist_final = _final_model.fit(
        _tr_gen_f,
        validation_data = _va_gen_f,
        epochs          = FINAL_EPOCHS,
        callbacks       = [EarlyStopping(
            monitor='val_loss', patience=FINAL_PATIENCE,
            restore_best_weights=True, verbose=1,
        )],
        verbose=1,
    )
    del _tr_gen_f, _va_gen_f;  gc.collect()

    # ── Evaluation — batched to avoid GPU OOM on 3.28 GB X_test ─────────────
    print('  Evaluating on test set (batched) ...')
    _test_gen_opt = SequenceDataGenerator(
        X_test, y_test, batch_size=FINAL_BATCH_SIZE, shuffle=False
    )
    _y_pred_raw = _final_model.predict(
        _test_gen_opt, verbose=1
    ).flatten()[:len(y_test)]

    # Inverse transform to original kWh space
    _y_true_kwh = np.expm1(np.clip(y_test.astype(np.float64), -10., 30.))
    _y_pred_kwh = np.maximum(
        np.expm1(np.clip(_y_pred_raw.astype(np.float64), -10., 30.)), 0.
    )

    # Compute and save metrics (already in kWh — log_transform=False)
    _m = save_benchmark_results(
        model_name     = 'drnn_optuna',
        horizon        = HORIZON,
        y_true         = _y_true_kwh,
        y_pred         = _y_pred_kwh,
        benchmark_root = BENCHMARK_DIR,
        log_transform  = False,   # already inverted above
        extra_meta     = {
            'best_params'   : to_python_types(_best_p),
            'best_val_loss' : float(_study.best_value),
            'n_trials'      : N_TRIALS,
            'sampler'       : 'TPE',
            'epochs_trained': len(_hist_final.history['loss']),
        },
    )
    all_results[HORIZON] = _m

    # ── Save model, scaler, training history, config ─────────────────────────
    _final_model.save(os.path.join(_opt_dir, f'drnn_optuna_k{HORIZON}.h5'))
    joblib.dump(scaler, os.path.join(_opt_dir, 'scaler.pkl'))

    with open(os.path.join(_opt_dir, f'training_history_k{HORIZON}.json'), 'w') as _f:
        json.dump(
            {k: [float(v) for v in vals]
             for k, vals in _hist_final.history.items()}, _f, indent=2
        )

    with open(os.path.join(_opt_dir, 'training_config.json'), 'w') as _f:
        json.dump({
            'horizon_k'       : HORIZON,
            'log_transform'   : TARGET_LOG_TRANSFORM,
            'loss_function'   : LOSS_FUNCTION,
            'huber_delta'     : HUBER_DELTA,
            'n_trials'        : N_TRIALS,
            'opt_subset_ratio': OPT_SUBSET_RATIO,
            'opt_time_limit'  : OPT_TIME_LIMIT,
            'final_batch_size': FINAL_BATCH_SIZE,
            'final_epochs'    : FINAL_EPOCHS,
            'final_patience'  : FINAL_PATIENCE,
            'random_seed'     : RANDOM_SEED,
            'best_params'     : to_python_types(_best_p),
            'best_val_loss'   : float(_study.best_value),
            'metrics'         : _m,
            'completed_at'    : datetime.now().isoformat(),
        }, _f, indent=4)

    # ── Plots (all in kWh space) ─────────────────────────────────────────────
    # Optuna convergence
    _tv   = [t.value for t in _study.trials if t.value is not None]
    _best = np.minimum.accumulate(_tv)
    _fig, _ax = plt.subplots(figsize=(8, 4))
    _ax.plot(_tv,   alpha=0.5, color='grey',      label='Trial val loss')
    _ax.plot(_best, lw=2,      color='darkorange', label='Running best')
    _ax.set_xlabel('Trial')
    _ax.set_ylabel(f'Validation {LOSS_FUNCTION.upper()} loss')
    _ax.set_title(f'Optuna Convergence  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'optuna_convergence.png'), dpi=200)
    plt.close('all')

    # Training loss curve
    _fig, _ax = plt.subplots(figsize=(8, 4))
    _ax.plot(_hist_final.history['loss'],     lw=1.5, label='Train loss')
    _ax.plot(_hist_final.history['val_loss'], lw=1.5, label='Val loss')
    _ax.set_xlabel('Epoch')
    _ax.set_ylabel(f'{LOSS_FUNCTION.upper()} loss')
    _ax.set_title(f'DRNN+Optuna Training  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'training_curve.png'), dpi=200)
    plt.close('all')

    # Predicted vs actual scatter (kWh)
    _fig, _ax = plt.subplots(figsize=(6, 6))
    _ax.scatter(_y_true_kwh, _y_pred_kwh, alpha=0.4, s=5, color='steelblue')
    _lim = [min(_y_true_kwh.min(), _y_pred_kwh.min()),
            max(_y_true_kwh.max(), _y_pred_kwh.max())]
    _ax.plot(_lim, _lim, 'r--', lw=1.5, label='Perfect')
    _ax.set_xlabel(f'Actual ΔE  k={HORIZON}s (kWh)')
    _ax.set_ylabel(f'Predicted ΔE  k={HORIZON}s (kWh)')
    _ax.set_title(f'DRNN+Optuna Predicted vs Actual  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'pred_vs_actual.png'), dpi=200)
    plt.close('all')

    # Residual distribution (kWh)
    _fig, _ax = plt.subplots(figsize=(7, 4))
    sns.histplot(_y_true_kwh - _y_pred_kwh, bins=60, kde=True,
                 ax=_ax, color='steelblue')
    _ax.axvline(0, color='red', lw=1.2, linestyle='--')
    _ax.set_xlabel('Residual (kWh)')
    _ax.set_title(f'DRNN+Optuna Residuals  k={HORIZON}s')
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'residuals.png'), dpi=200)
    plt.close('all')

    # Time-series overlay (first 500 test samples, kWh)
    _ns = min(500, len(_y_true_kwh))
    _fig, _ax = plt.subplots(figsize=(12, 4))
    _ax.plot(_y_true_kwh[:_ns], lw=1.0, alpha=0.9,  label='Actual')
    _ax.plot(_y_pred_kwh[:_ns], lw=1.0, alpha=0.75,
             linestyle='--', label='Predicted')
    _ax.set_xlabel('Test sample index')
    _ax.set_ylabel(f'ΔE  k={HORIZON}s (kWh)')
    _ax.set_title(f'First {_ns} test samples — DRNN+Optuna  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'time_series_overlay.png'), dpi=200)
    plt.close('all')

    print(f'  Plots → {_pd}')
    print(f'  MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  '
          f'R2={_m["R2"]:.4f}  sMAPE={_m["sMAPE"]:.4f}')

    del _final_model, _hist_final, _y_true_kwh, _y_pred_kwh, _y_pred_raw
    del X_train, y_train, X_val, y_val, X_test, y_test, scaler
    K.clear_session();  gc.collect()

    print(f'\n  DONE k={HORIZON}s')
    print('=' * 60)

_total_h = (time.time() - loop_start) / 3600
print(f'\nDRNN+Optuna complete — {_total_h:.2f} h total')
if all_results:
    print(f'\n{"k (s)":>8} {"MAE":>12} {"RMSE":>12} {"R2":>8} {"sMAPE":>10}')
    print('-' * 56)
    for _hk, _r in sorted(all_results.items()):
        print(f'{_hk:>8} {_r["MAE"]:>12.6f} {_r["RMSE"]:>12.6f} '
              f'{_r["R2"]:>8.4f} {_r["sMAPE"]:>10.4f}')

---
## Cross-Horizon Summary

In [ ]:
if not all_results:
    print('No results yet — run the main loop first.')
else:
    _rows = [{'Horizon_k_s': _h, 'Model': 'drnn_optuna', **_m}
             for _h, _m in sorted(all_results.items())]
    _df = pd.DataFrame(_rows)
    print('DRNN + Optuna — all horizons')
    print('=' * 72)
    print(_df.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 72)
    _csv = os.path.join(ANALYSIS_DIR, 'drnn_optuna_all_horizons.csv')
    _df.to_csv(_csv, index=False)
    print(f'Saved → {_csv}')